©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、勾配降下法の仕組みをPythonで実装して学びます。数値微分による勾配計算関数を作成し、2変数関数に対して勾配降下法を適用することで、パラメータが損失関数の最小値に向かって更新されていく過程を数値的・視覚的に確認します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）


In [ ]:
# %%capture
# !pip uninstall matplotlib -y
# !pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

# !pip uninstall torch -y
# !pip install torch==2.7.0

# !pip uninstall torchvision -y
# !pip install torchvision==0.22.0

# 勾配降下法

In [ ]:
# ライブラリのインポート
import numpy as np

## 勾配の計算

In [ ]:
# 利用する関数の定義
def function_2(x):
    return x[0]**2 + x[1]**2

In [ ]:
# 勾配の数値計算
def numerical_gradient(f, x):
    """
    数値微分を用いて勾配を計算する関数
    中心差分近似を使用して、各パラメータに対する勾配を数値的に求める

    主な用途:
    - 解析的に求めた勾配（逆伝播）の正しさを検証する（勾配チェック）
    - 微分が複雑で解析的に求めにくい場合の代替手段

    注意: 計算コストが高いため、実際の学習には使用せず、デバッグ目的で使用

    Args:
        f: 勾配を求めたい関数（損失関数など）
        x: 勾配を計算する地点のパラメータ（numpy配列）

    Returns:
        grad: xの各要素に対する勾配（xと同じ形状のnumpy配列）
    """
    h = 1e-4           # 微小な変化量（小さすぎると丸め誤差、大きすぎると近似精度が落ちる）

    # 勾配を格納する配列（xと同じ形状でゼロ初期化）
    grad = np.zeros_like(x)

    # xの各要素について勾配を計算
    for idx in range(x.size):
        tmp_val = x[idx]      # 元の値を保存

        # f(x+h)の計算
        # xのidx番目の要素だけをhだけ増やす
        x[idx] = tmp_val + h
        fxh1 = f(x)      # x+hでの関数値

        # f(x-h)の計算
        # xのidx番目の要素だけをhだけ減らす
        x[idx] = tmp_val - h
        fxh2 = f(x)      # x-hでの関数値

        # 中心差分近似で勾配を計算
        # 導関数の定義: lim(h→0) [f(x+h) - f(x-h)] / (2h)
        # 中心差分は前進差分や後退差分より精度が高い（誤差がO(h^2)）
        grad[idx] = (fxh1 - fxh2) / (2*h)

        # 値を元に戻す（次の要素の計算のため）
        x[idx] = tmp_val

    return grad

### 勾配の実数値

In [ ]:
# function_2の勾配を点(3.0, 4.0)で数値微分により計算
# numerical_gradient関数を使って、各変数に対する偏微分を数値的に求める
numerical_gradient(function_2, np.array([3.0, 4.0]))

array([6., 8.])

In [ ]:
# function_2の勾配を点(0.0, 2.0)で数値微分により計算
# x[0]=0.0, x[1]=2.0 における各変数に対する偏微分を数値的に求める
numerical_gradient(function_2, np.array([0.0, 2.0]))

array([0., 4.])

In [ ]:
# function_2の勾配を点(3.0, 0.0)で数値微分により計算
# x[0]=3.0, x[1]=0.0 における各変数に対する偏微分を数値的に求める
numerical_gradient(function_2, np.array([3.0, 0.0]))

array([6., 0.])

## 勾配降下法

In [ ]:
# 勾配降下法
def gradient_descent(f, init_x, lr=0.01, step_num=100):
    """
    勾配降下法を用いて関数の最小値を求める最適化アルゴリズム

    勾配降下法の原理:
    - 勾配（gradient）は関数が最も急激に増加する方向を示す
    - その逆方向（-gradient）に進むことで、関数値を減少させる
    - 学習率（learning rate）で更新幅を調整する

    Args:
        f: 最小化したい目的関数（損失関数など）
        init_x: 初期値（探索の開始点）
        lr: 学習率（learning rate）- 1回の更新でどれだけ移動するか
            - 大きすぎると発散する可能性がある
            - 小さすぎると収束が遅い
        step_num: 更新回数（イテレーション数）

    Returns:
        x: 最適化後のパラメータ（関数の最小値付近の点）
    """
    x = init_x    # 現在の位置を初期値で初期化

    # 指定された回数だけ更新を繰り返す
    for i in range(step_num):
        # 現在の位置xでの勾配を数値微分で計算
        # 勾配は関数が増加する方向を示すベクトル
        grad = numerical_gradient(f, x)

        # パラメータの更新: x = x - η * ∇f(x)
        # η（イータ）は学習率（lr）
        # ∇f(x)は勾配（grad）
        # マイナスをつけることで、勾配の逆方向（降下方向）に移動
        x -= lr * grad

    return x

In [ ]:
# 初期値の設定: 点(-3.0, 4.0)から探索を開始
init_x = np.array([-3.0, 4.0])

# 勾配降下法を実行して、function_2の最小値を求める
# init_x: 初期位置 (-3.0, 4.0)
# lr=0.1: 学習率（各ステップでの移動量の係数）
# step_num=100: 100回の更新を行う
gradient_descent(function_2, init_x=init_x, lr=0.1, step_num=100)

array([-6.11110793e-10,  8.14814391e-10])

In [ ]:
# 学習率が大きすぎる例（不適切なハイパーパラメータの設定）
# 学習率が大きすぎると、最小値を飛び越えて発散する可能性がある

# 初期値の設定
init_x = np.array([-3.0, 4.0])

# 勾配降下法を実行（学習率が大きすぎる）
# lr=10.0: 学習率が非常に大きい（通常は0.001～0.1程度が適切）
# この設定では、更新幅が大きすぎて最小値に収束せず発散する
gradient_descent(function_2, init_x=init_x, lr=10, step_num=100)

array([-2.58983747e+13, -1.29524862e+12])

In [ ]:
# 学習率が小さすぎる例（不適切なハイパーパラメータの設定）
# 学習率が小さすぎると、最小値に向かって進むが収束が極めて遅い

# 初期値の設定
init_x = np.array([-3.0, 4.0])

# 勾配降下法を実行（学習率が小さすぎる）
# lr=1e-10: 学習率が極端に小さい（0.0000000001）
# この設定では、更新幅が小さすぎてほとんど動かず、100回の更新では不十分
gradient_descent(function_2, init_x=init_x, lr=1e-10, step_num=100)

array([-2.99999994,  3.99999992])